# nb10.5 - Transportability Audit: Will Your Estimate Hold in a Different Population?

*Week 10, Chapter 10.1 companion. Leah has a clean ATT from a patent reform that affected
biotech firms in 2018-2022. The discussant asks: "Does this generalize to the manufacturing sector
in 2024?" Two distinct populations, possibly different covariate distributions, possibly different
effect-modifier weights. Internal validity won the first battle; external validity is the next one.*

The reveal-the-trick lesson: **transportability is a covariate-distribution problem before it is a
heterogeneity problem.** If the study population has different distributions of the covariates that
modify the treatment effect, then even a perfectly identified ATT in the study sample need not equal
the ATT in the target population. The fix - when covariates overlap enough - is to **reweight** the
study sample so its covariate distribution matches the target's, then recompute the ATT on the
reweighted study. If the reweighted ATT differs sharply from the unweighted, the result *does not
transport*. The cleanest weight is the **density ratio** $w(\mathbf{x}) = f_T(\mathbf{x}) /
f_S(\mathbf{x})$, where $S$ is the study population and $T$ is the target.

We will simulate two populations with the same individual-level structural treatment effect but
different covariate distributions. We will estimate the density ratio via a probabilistic classifier
(study-vs-target), reweight, and report a **bound on the transport error** of the original ATT.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. Two populations, one structural model

We build a study population $S$ of $N_S = 2000$ firms and a target population $T$ of $N_T = 2000$
firms. Covariates are $\mathbf{x} = (x_1, x_2)$. The study has $\mathbf{x} \sim \mathcal{N}(0, I)$;
the target has $\mathbf{x} \sim \mathcal{N}((0.6, -0.4), \Sigma)$ with $\Sigma$ a mildly correlated
matrix. The treatment $D$ is randomized in both populations at $\Pr(D=1) = 0.5$. The structural
treatment effect is heterogeneous and identical across populations:
$$\tau(\mathbf{x}) = 0.6 + 0.8 x_1 - 0.5 x_2.$$
Because the covariate distributions differ, the *population ATT* differs even though every individual
gets the same effect.

In [ ]:
def draw_study(rng, N=2000):
    X = rng.normal(0, 1, size=(N, 2))
    D = rng.binomial(1, 0.5, size=N)
    return X, D

def draw_target(rng, N=2000):
    mean = np.array([0.6, -0.4])
    L = np.array([[1.0, 0.0], [0.4, 0.9]])         # cholesky-like
    Z = rng.normal(0, 1, size=(N, 2))
    X = mean + Z @ L.T
    D = rng.binomial(1, 0.5, size=N)
    return X, D

def tau_x(X):
    return 0.6 + 0.8 * X[:,0] - 0.5 * X[:,1]

def Y_outcome(rng, X, D):
    # Baseline + heterogeneous treatment effect + noise
    g = 0.4 * X[:,0] + 0.2 * X[:,1]
    eps = rng.normal(0, 1, size=len(X))
    return g + tau_x(X) * D + eps

Xs, Ds = draw_study(rng, N=2000)
Ys = Y_outcome(rng, Xs, Ds)

Xt, Dt = draw_target(rng, N=2000)
Yt = Y_outcome(rng, Xt, Dt)

study  = pd.DataFrame(Xs, columns=["x1","x2"]); study["D"] = Ds; study["Y"] = Ys
target = pd.DataFrame(Xt, columns=["x1","x2"]); target["D"] = Dt; target["Y"] = Yt

print(f"study  covariate means: x1={study['x1'].mean():0.3f}, x2={study['x2'].mean():0.3f}")
print(f"target covariate means: x1={target['x1'].mean():0.3f}, x2={target['x2'].mean():0.3f}")
print(f"true ATT in study : {tau_x(Xs[Ds==1]).mean():0.3f}")
print(f"true ATT in target: {tau_x(Xt[Dt==1]).mean():0.3f}")

**Read the printout.** The two populations have identical individual-level treatment effects -
every firm with the same $\mathbf{x}$ gets the same $\tau$. But because the target population is
shifted in covariate space (higher $x_1$, lower $x_2$, with $x_2$ pulling the effect *up*), the
target ATT is larger than the study ATT. A naive transport - reporting the study ATT as if it applied
to the target - would understate the policy impact.

## 2. Estimate the ATT in the study (the number we want to transport)

In [ ]:
def estimate_att(df):
    # Simple diff-in-means + a covariate-adjusted OLS as cross-check.
    dim = df.loc[df["D"]==1, "Y"].mean() - df.loc[df["D"]==0, "Y"].mean()
    ols = sm.OLS(df["Y"].values,
                 sm.add_constant(df[["D","x1","x2"]].values)).fit()
    return dim, ols.params[1]

att_study_dim, att_study_ols = estimate_att(study)
att_target_dim, att_target_ols = estimate_att(target)
print(f"study  ATT  (dim) = {att_study_dim:0.3f}  (OLS) = {att_study_ols:0.3f}")
print(f"target ATT  (dim) = {att_target_dim:0.3f}  (OLS) = {att_target_ols:0.3f}")

**The transport problem in one line.** We have $\widehat{\text{ATT}}_S \approx 0.6$ but the
target ATT is $\approx 1.3$. If we report the study number to a target-population audience we are
**half-right**: same sign, but the magnitude is wrong by a factor of two. We need to do something
better.

## 3. Density-ratio reweighting

The trick: stack the study sample (labeled $Z = 0$) and a *sample from the target's covariate
distribution* (labeled $Z = 1$). Train a probabilistic classifier $\Pr(Z = 1 \mid \mathbf{x})$. Then
$$\frac{f_T(\mathbf{x})}{f_S(\mathbf{x})} = \frac{\Pr(Z=1 \mid \mathbf{x}) / \Pr(Z=1)}
                                         {\Pr(Z=0 \mid \mathbf{x}) / \Pr(Z=0)}.$$
When the marginals are balanced ($\Pr(Z=1) = \Pr(Z=0) = 0.5$, which happens automatically if we draw
equal-size pools), this simplifies to $f_T / f_S = p / (1 - p)$ where $p$ is the predicted probability
of belonging to the target. The reweighted study moments approximate the target moments, and so does
the reweighted ATT.

In [ ]:
def estimate_density_ratio(study_X, target_X, seed=7):
    Xstack = np.vstack([study_X, target_X])
    Z = np.concatenate([np.zeros(len(study_X)), np.ones(len(target_X))]).astype(int)
    # Gradient boosting captures non-linear differences cheaply.
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                     random_state=seed).fit(Xstack, Z)
    p = clf.predict_proba(study_X)[:, 1]      # P(Z=1 | x) for each study row
    p = np.clip(p, 0.02, 0.98)
    w = p / (1 - p)                            # density ratio f_T / f_S
    return w, clf

w, clf = estimate_density_ratio(Xs, Xt)
print(f"weight stats: mean={w.mean():0.3f}, sd={w.std():0.3f}, min={w.min():0.3f}, max={w.max():0.3f}")
print(f"covariate means BEFORE reweighting: x1={Xs[:,0].mean():0.3f}, x2={Xs[:,1].mean():0.3f}")
print(f"covariate means AFTER  reweighting: x1={(Xs[:,0]*w).sum()/w.sum():0.3f}, "
      f"x2={(Xs[:,1]*w).sum()/w.sum():0.3f}")
print(f"target covariate means              : x1={Xt[:,0].mean():0.3f}, x2={Xt[:,1].mean():0.3f}")

**Read the means.** After applying the density-ratio weights, the *study* covariate means
should sit close to the *target* means. They do. The weights tilt mass toward study units that look
like target units (positive $x_1$, negative $x_2$) and away from units that don't.

## 4. Reweighted ATT estimate

In [ ]:
def att_reweighted(df, w):
    # weighted diff-in-means + weighted OLS
    treated = df["D"]==1; control = df["D"]==0
    w_t = w[treated]; w_c = w[control]
    Y = df["Y"].values
    mean_t = (Y[treated] * w_t).sum() / w_t.sum()
    mean_c = (Y[control] * w_c).sum() / w_c.sum()
    dim = mean_t - mean_c
    ols = sm.WLS(df["Y"].values,
                 sm.add_constant(df[["D","x1","x2"]].values),
                 weights=w).fit()
    return dim, ols.params[1]

att_rw_dim, att_rw_ols = att_reweighted(study, w)
print(f"reweighted study ATT (dim) = {att_rw_dim:0.3f}  (OLS) = {att_rw_ols:0.3f}")
print(f"target           ATT (dim) = {att_target_dim:0.3f}  (OLS) = {att_target_ols:0.3f}")
print(f"truth in target            = {tau_x(Xt[Dt==1]).mean():0.3f}")

**The win.** The reweighted study ATT lands much closer to the target ATT than the unweighted
one did. The transport gap has shrunk from $\sim 0.7$ down to $\sim 0.1$ or so. We have actually moved
the estimate to the population we want it to describe - assuming the density-ratio model is correct.

## 5. A picture of the reweighting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(Xs[:,0], Xs[:,1], s=8, alpha=0.3, label="study (uniform weight)")
axes[0].scatter(Xt[:,0], Xt[:,1], s=8, alpha=0.3, color="C3", label="target")
axes[0].set_xlabel("$x_1$"); axes[0].set_ylabel("$x_2$"); axes[0].legend()
axes[0].set_title("Raw covariate clouds")

# Color study points by their weight: high weight = redder
axes[1].scatter(Xt[:,0], Xt[:,1], s=8, alpha=0.15, color="C3", label="target")
sc = axes[1].scatter(Xs[:,0], Xs[:,1], s=8, c=np.log(w), alpha=0.6, cmap="viridis")
plt.colorbar(sc, ax=axes[1], label="log weight")
axes[1].set_xlabel("$x_1$"); axes[1].set_ylabel("$x_2$"); axes[1].legend()
axes[1].set_title("Study points coloured by density-ratio weight")
plt.tight_layout(); plt.close(fig)
"plotted"


**The reweighting in pictures.** Study points that lie inside the target cloud get high weights
(brighter colours). Study points far from the target cloud get tiny weights. This is exactly the kind
of figure a discussant wants to see: it tells them how much of the study really speaks to the target
question.

## 6. Bounding the transport error

Even if the density-ratio weights look beautiful, two failure modes survive: (1) the classifier may be
misspecified, so the weights are off; (2) parts of target-covariate space may have *zero* support in
the study (non-overlap). A clean way to bound the transport error is to use the **effective sample
size** (Kish) and the **maximum weight**.

The Kish effective sample size is
$$N_{\text{eff}} = \frac{(\sum w_i)^2}{\sum w_i^2}.$$
If $N_{\text{eff}} \ll N_S$, the reweighting is leaning hard on a few units and its variance is large.
A simple, conservative transport-error bound is
$$|\widehat\tau_T - \widehat\tau_S^{\text{rw}}| \lesssim
   c \cdot \frac{\max_i w_i}{N_{\text{eff}}^{1/2}} \cdot \sigma_Y,$$
with $c$ a tuning constant (we use $c = 2$ for the 95% band of a Gaussian-ish residual). The
formula is a back-of-envelope, not a sharp theorem; the point is to surface the *brittleness* of the
reweighting in a single scalar.

In [ ]:
def transport_bound(w, Y, c=2.0):
    Neff = (w.sum() ** 2) / (w ** 2).sum()
    sigma = Y.std()
    return c * w.max() / np.sqrt(Neff) * sigma, Neff

bound, Neff = transport_bound(w, study["Y"].values)
print(f"N_eff (Kish)      = {Neff:0.1f}  (versus N_S = {len(study)})")
print(f"transport bound   ~ {bound:0.3f}")
print(f"observed gap |target ATT - reweighted study ATT| = "
      f"{abs(att_target_dim - att_rw_dim):0.3f}")

**Read the bound.** If $N_{\text{eff}}$ collapses much below $N_S$, the bound widens fast - a
visual warning that the reweighting has paid for itself in variance. In our DGP the populations are
close enough that $N_{\text{eff}}$ remains a healthy fraction of $N_S$ and the bound is tight.

## 7. Monte Carlo: when does transport go bad?

We sweep the **target shift** parameter (how far the target population is shifted in $x_1$) and watch
how the unweighted gap, the reweighted gap, and the transport bound move together.

In [ ]:
def one_transport_sim(seed, shift_x1=0.6, shift_x2=-0.4):
    rng_s = np.random.default_rng(seed)
    Xs2, Ds2 = draw_study(rng_s, N=1500)
    Ys2 = Y_outcome(rng_s, Xs2, Ds2)
    # custom target with adjustable shift
    mean = np.array([shift_x1, shift_x2])
    L = np.array([[1.0, 0.0], [0.4, 0.9]])
    Z = rng_s.normal(0, 1, size=(1500, 2))
    Xt2 = mean + Z @ L.T
    Dt2 = rng_s.binomial(1, 0.5, size=1500)
    Yt2 = Y_outcome(rng_s, Xt2, Dt2)
    s2 = pd.DataFrame(Xs2, columns=["x1","x2"]); s2["D"] = Ds2; s2["Y"] = Ys2
    t2 = pd.DataFrame(Xt2, columns=["x1","x2"]); t2["D"] = Dt2; t2["Y"] = Yt2
    att_s, _ = estimate_att(s2)
    att_t, _ = estimate_att(t2)
    w2, _ = estimate_density_ratio(Xs2, Xt2, seed=seed+1)
    att_rw, _ = att_reweighted(s2, w2)
    bound2, _ = transport_bound(w2, s2["Y"].values)
    return {"unweighted_gap": att_t - att_s,
            "reweighted_gap": att_t - att_rw,
            "bound": bound2}

shifts = [0.0, 0.3, 0.6, 1.0, 1.5, 2.0]
rows = []
for s in shifts:
    rec = one_transport_sim(seed=999, shift_x1=s, shift_x2=-0.4)
    rec["shift_x1"] = s
    rows.append(rec)
sweep_df = pd.DataFrame(rows)
sweep_df

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep_df["shift_x1"], sweep_df["unweighted_gap"], marker="o", label="unweighted gap")
ax.plot(sweep_df["shift_x1"], sweep_df["reweighted_gap"], marker="s", label="reweighted gap")
ax.plot(sweep_df["shift_x1"], sweep_df["bound"],          marker="^", label="transport bound", ls="--")
ax.axhline(0, color="grey", lw=0.6)
ax.set_xlabel(r"shift of target population in $x_1$")
ax.set_ylabel("ATT gap")
ax.set_title("Reweighting helps - but only inside the overlap region")
ax.legend()
plt.tight_layout(); plt.close(fig)
"plotted"


**Read the plot.** For small shifts the reweighted gap is near zero and the bound is tight.
As the shift grows, the overlap between study and target shrinks; the density-ratio weights become
extreme; the bound balloons. The reweighting *cannot* save you when there is no support: a
density-ratio estimate of "this region of target space" requires some study observations there.

This is the takeaway that belongs in your paper's external-validity section: **report the unweighted
gap, the reweighted gap, and the Kish effective sample size, side by side.** A reader who sees a tiny
$N_{\text{eff}}$ should distrust the reweighted number, no matter how close to the target ATT it
happens to land.

## 8. When the audit fails (be honest)

1. **No overlap.** If the target distribution puts mass where the study distribution does not, the
   density ratio is technically infinite and no reweighting recovers the target ATT. Trim or
   pre-register a smaller target region.

2. **Effect-modifier confounding outside the observed $\mathbf{x}$.** The audit assumes the
   *modifiers of treatment heterogeneity* are all in $\mathbf{x}$. If there are unobserved
   modifiers, the reweighting matches observed covariates but not the actual effect heterogeneity.
   The result transports the wrong number.

3. **Tiny target sample.** The density-ratio classifier needs target observations. If your target
   sample is small ($N_T < 200$), the classifier overfits and the weights become noisy.

4. **Mis-specified classifier.** Linear logistic may miss non-linear differences; tree-based
   classifiers may overfit on rare combinations. The standard practice (Pearl & Bareinboim 2014) is
   to report results under *two* classifier families and check for stability.

## 9. Your turn

1. **Shift the target away.** Push `shift_x1` to 3.0 and rerun. The density-ratio classifier will
   start producing extreme weights; the bound will balloon. Plot a histogram of the weights.

2. **Misspecify the modifier.** Add a third covariate $x_3$ to the DGP, make it an effect modifier
   (multiply the treatment effect by $0.5 x_3$), but pretend you never observed $x_3$ when you
   reweight. Does the reweighting still work?

3. **Pearl-Bareinboim diagram.** Sketch the selection DAG implied by this DGP (the *S-admissibility*
   condition of Pearl & Bareinboim 2014). Mark which arrows your $\mathbf{x}$ blocks and which you
   are assuming away.

**Citations.** Pearl & Bareinboim (2014, *Statistical Science* 29(4):579-595, "External Validity:
From Do-Calculus to Transportability Across Populations"); Bareinboim & Pearl (2016, *PNAS*
113(27):7345-7352); Stuart, Cole, Bradshaw & Leaf (2011, *JRSS-A* 174(2):369-386); Hartman, Grieve,
Ramsahai & Sekhon (2015, *JRSS-A* 178(3):757-778).